# Notebook 28: LoRA Merging - Multi-Adapter Composition

**Learning Objectives:**
- Learn how to combine multiple LoRA adapters
- Understand task arithmetic (add, subtract, average adapters)
- Implement weighted merging for multi-task models
- Compare sequential vs parallel adapter application
- Merge LoRA back to base model for deployment
- Create "model soups" with LoRA
- Design multi-adapter inference strategies

**Prerequisites:** Notebooks 23-27 (LoRA fundamentals, PEFT comparison)

## 1. Why Merge LoRA Adapters?

### Use Cases for LoRA Merging

**1. Deployment Optimization**
```
Problem: Runtime adapter application adds overhead
  Base: W
  Forward: (W + BA) @ x requires storing B, A separately
  
Solution: Merge adapter into base weights
  W' = W + BA
  Forward: W' @ x (same as non-LoRA model)
  
Result: Zero inference overhead!
```

**2. Multi-Task Composition**
```
Scenario: Train 3 LoRA adapters for different tasks
  - Adapter_math: Mathematical reasoning
  - Adapter_code: Code generation  
  - Adapter_writing: Creative writing
  
Goal: Create multi-task model good at all three

Approach: Weighted merge
  W' = W + α*B_math@A_math + β*B_code@A_code + γ*B_writing@A_writing
```

**3. Task Arithmetic**
```
Add task: W' = W + BA_new
Remove task: W' = W - BA_old
Average tasks: W' = W + (1/n)Σ(B_i @ A_i)
Interpolate: W' = W + α*BA_task1 + (1-α)*BA_task2
```

**4. Model Soups**
```
Train same task with different:
  - Random seeds
  - Hyperparameters
  - Data augmentations
  
Merge adapters → often better than best individual!
```

**5. Continual Learning**
```
Learn task 1 → Adapter A1
Learn task 2 → Adapter A2
Learn task 3 → Adapter A3

Merge all → retain knowledge from all tasks
Avoids catastrophic forgetting
```

In [ ]:
# Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Optional, Tuple
import copy

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## 2. Merging LoRA into Base Model (Deployment)

### The Mathematics
```
During training:
  Original: h = W @ x
  With LoRA: h = (W + BA) @ x = W @ x + BA @ x
  
  W is frozen, B and A are trained

After training (merge):
  W_merged = W + s * BA
  where s is scaling factor (typically s = α/r)
  
  Forward: h = W_merged @ x
  
Result: Same outputs, but only single weight matrix!
```

### Implementation Details
```python
# During training
class LoRALayer:
    self.weight = W          # Frozen (d_out, d_in)
    self.lora_A = A          # Trainable (d_in, r)
    self.lora_B = B          # Trainable (r, d_out)
    self.scaling = alpha / r
    
    def forward(x):
        # Base computation
        result = F.linear(x, self.weight)
        # LoRA addition
        result += (x @ self.lora_A @ self.lora_B.T) * self.scaling
        return result

# For deployment (merge)
def merge_lora():
    # Compute ΔW = BA^T
    delta_w = (self.lora_B.T @ self.lora_A.T) * self.scaling
    # Add to base weights
    self.weight.data += delta_w
    # Can now delete A and B!
    del self.lora_A
    del self.lora_B
```

In [ ]:
class LoRALinear(nn.Module):
    """Linear layer with LoRA adaptation"""
    
    def __init__(
        self,
        in_features: int,
        out_features: int,
        rank: int = 8,
        alpha: float = 16.0,
        bias: bool = True
    ):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.scaling = alpha / rank
        
        # Base layer (frozen)
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.02)
        self.weight.requires_grad = False
        
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features))
            self.bias.requires_grad = False
        else:
            self.bias = None
        
        # LoRA parameters (trainable)
        self.lora_A = nn.Parameter(torch.randn(in_features, rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features))
        
        self.merged = False
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (batch, seq_len, in_features)
        Returns:
            (batch, seq_len, out_features)
        """
        # Base computation
        result = F.linear(x, self.weight, self.bias)
        
        if not self.merged:
            # Add LoRA adaptation
            # x @ A @ B^T = x @ (A @ B^T)
            lora_out = x @ self.lora_A @ self.lora_B
            result = result + lora_out * self.scaling
        
        return result
    
    def merge_lora(self):
        """Merge LoRA weights into base weights for deployment"""
        if self.merged:
            print("LoRA already merged")
            return
        
        # Compute ΔW = (B^T @ A^T) * scaling
        # Note: weight is (out, in), so we need B^T @ A^T
        with torch.no_grad():
            delta_w = (self.lora_B.T @ self.lora_A.T) * self.scaling
            self.weight.data += delta_w
        
        self.merged = True
        print(f"Merged LoRA into base weights. ΔW norm: {delta_w.norm().item():.4f}")
    
    def unmerge_lora(self):
        """Unmerge LoRA weights (reverse merge)"""
        if not self.merged:
            print("LoRA not merged")
            return
        
        with torch.no_grad():
            delta_w = (self.lora_B.T @ self.lora_A.T) * self.scaling
            self.weight.data -= delta_w
        
        self.merged = False
        print("Unmerged LoRA from base weights")


# Test merge functionality
layer = LoRALinear(512, 512, rank=8, alpha=16.0)

# Create test input
x = torch.randn(4, 32, 512)

print("Testing LoRA merge functionality:\n")
print("1. Forward pass with separate LoRA")
output_before = layer(x)
print(f"   Output shape: {output_before.shape}")
print(f"   Output mean: {output_before.mean().item():.6f}")

print("\n2. Merge LoRA into base weights")
layer.merge_lora()

print("\n3. Forward pass with merged weights")
output_after = layer(x)
print(f"   Output shape: {output_after.shape}")
print(f"   Output mean: {output_after.mean().item():.6f}")

print("\n4. Verify outputs match")
diff = (output_before - output_after).abs().max().item()
print(f"   Max difference: {diff:.10f}")
print(f"   Match: {diff < 1e-5}")

print("\n5. Unmerge and verify")
layer.unmerge_lora()
output_unmerged = layer(x)
diff2 = (output_before - output_unmerged).abs().max().item()
print(f"   Max difference from original: {diff2:.10f}")
print(f"   Match: {diff2 < 1e-5}")

## 3. Task Arithmetic: Add, Subtract, Average Adapters

### Theoretical Foundation
```
Key insight: LoRA adapters are additive!
  W_base + BA_task1 + BA_task2 = W_base + (BA_task1 + BA_task2)

This means we can treat adapters as vectors in weight space
and perform linear operations:
```

### Operations

**1. Addition (Multi-task learning)**
```
W_multi = W_base + BA_task1 + BA_task2 + BA_task3

Effect: Model gains capabilities from all tasks
Caveat: Tasks might interfere if conflicting
```

**2. Subtraction (Remove unwanted behavior)**
```
W_fixed = W_base + BA_good - BA_bad

Example: Remove toxic language generation
  W' = W + BA_helpful - BA_toxic
  
Effect: Subtract unwanted task vector
```

**3. Averaging (Ensemble)**
```
W_ensemble = W_base + (1/n) * Σ(BA_i)

Effect: Smooth out individual model quirks
Often more robust than single model
```

**4. Weighted Combination**
```
W_combined = W_base + Σ(α_i * BA_i)
where Σα_i = 1

Example: 70% math, 30% code
  W' = W + 0.7*BA_math + 0.3*BA_code
```

**5. Interpolation (Gradual transition)**
```
W_t = W_base + (1-t)*BA_task1 + t*BA_task2
where t ∈ [0, 1]

t=0: Full task1
t=0.5: Equal mix
t=1: Full task2
```

In [ ]:
class MultiLoRALinear(nn.Module):
    """Linear layer with multiple LoRA adapters for task arithmetic"""
    
    def __init__(
        self,
        in_features: int,
        out_features: int,
        rank: int = 8,
        alpha: float = 16.0,
        num_adapters: int = 3
    ):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.num_adapters = num_adapters
        self.scaling = alpha / rank
        
        # Base layer (frozen)
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.02)
        self.weight.requires_grad = False
        self.bias = nn.Parameter(torch.zeros(out_features))
        self.bias.requires_grad = False
        
        # Multiple LoRA adapters
        self.lora_A = nn.ParameterList([
            nn.Parameter(torch.randn(in_features, rank) * 0.01)
            for _ in range(num_adapters)
        ])
        self.lora_B = nn.ParameterList([
            nn.Parameter(torch.zeros(rank, out_features))
            for _ in range(num_adapters)
        ])
    
    def forward(
        self, 
        x: torch.Tensor, 
        adapter_weights: Optional[List[float]] = None
    ) -> torch.Tensor:
        """
        Args:
            x: Input tensor
            adapter_weights: Weight for each adapter (if None, use all equally)
        """
        if adapter_weights is None:
            adapter_weights = [1.0] * self.num_adapters
        
        # Base computation
        result = F.linear(x, self.weight, self.bias)
        
        # Add weighted LoRA contributions
        for i, (A, B, weight) in enumerate(zip(self.lora_A, self.lora_B, adapter_weights)):
            if weight != 0:
                lora_out = x @ A @ B
                result = result + lora_out * self.scaling * weight
        
        return result
    
    def merge_adapters(
        self,
        adapter_weights: List[float],
        adapter_names: Optional[List[str]] = None
    ) -> torch.Tensor:
        """
        Merge multiple adapters with specified weights
        
        Returns:
            merged_delta: The combined ΔW from all adapters
        """
        if adapter_names is None:
            adapter_names = [f"Adapter {i}" for i in range(self.num_adapters)]
        
        merged_delta = torch.zeros_like(self.weight)
        
        print("Merging adapters:")
        for i, (A, B, weight, name) in enumerate(zip(
            self.lora_A, self.lora_B, adapter_weights, adapter_names
        )):
            delta = (B.T @ A.T) * self.scaling * weight
            merged_delta += delta
            print(f"  {name}: weight={weight:.2f}, norm={delta.norm().item():.4f}")
        
        print(f"Combined ΔW norm: {merged_delta.norm().item():.4f}")
        return merged_delta


# Demonstrate task arithmetic
layer = MultiLoRALinear(512, 512, rank=8, num_adapters=3)
x = torch.randn(4, 32, 512)

adapter_names = ['Math', 'Code', 'Writing']

print("Task Arithmetic Demonstrations:\n")
print("="*60)

# 1. Individual adapters
print("\n1. Individual Adapters:")
for i in range(3):
    weights = [0.0] * 3
    weights[i] = 1.0
    out = layer(x, weights)
    print(f"   {adapter_names[i]}: mean={out.mean().item():.4f}, std={out.std().item():.4f}")

# 2. Equal averaging
print("\n2. Equal Average (Model Soup):")
weights_avg = [1/3, 1/3, 1/3]
out_avg = layer(x, weights_avg)
print(f"   Mean: {out_avg.mean().item():.4f}, Std: {out_avg.std().item():.4f}")

# 3. Weighted combination
print("\n3. Weighted Combination (70% Math, 30% Code):")
weights_custom = [0.7, 0.3, 0.0]
out_custom = layer(x, weights_custom)
print(f"   Mean: {out_custom.mean().item():.4f}, Std: {out_custom.std().item():.4f}")

# 4. Subtraction (remove task)
print("\n4. Subtraction (Math + Code - Writing):")
weights_sub = [1.0, 1.0, -1.0]
out_sub = layer(x, weights_sub)
print(f"   Mean: {out_sub.mean().item():.4f}, Std: {out_sub.std().item():.4f}")

# 5. Merge analysis
print("\n5. Merge Analysis:")
print("\n   Scenario A: Equal weights")
delta_equal = layer.merge_adapters([1.0, 1.0, 1.0], adapter_names)

print("\n   Scenario B: Weighted merge")
delta_weighted = layer.merge_adapters([0.5, 0.3, 0.2], adapter_names)

print("\n   Scenario C: Subtraction")
delta_subtract = layer.merge_adapters([1.0, 1.0, -0.5], adapter_names)

## 4. Weighted Merging for Multi-Task Models

### Finding Optimal Weights

**Problem:**
```
Given: k task-specific LoRA adapters
Goal: Find weights α = [α1, α2, ..., αk] that maximize
      combined performance on all tasks

Constraint: Σ αi = 1 (typically)
```

**Approaches:**

1. **Uniform Weighting** (baseline)
   ```
   αi = 1/k for all i
   Simple, often works well
   ```

2. **Performance-Based Weighting**
   ```
   αi ∝ accuracy_i on validation set
   Give more weight to better-performing adapters
   ```

3. **Task-Specific Weighting**
   ```
   At inference: α = [1, 0, 0] for task 1
   Different weight vector per test sample
   ```

4. **Learned Weighting**
   ```
   Train small network to predict α given input
   α = MLP(input_embedding)
   Dynamic, input-dependent routing
   ```

5. **Grid Search / Bayesian Optimization**
   ```
   Search over α space to maximize validation metric
   More expensive but can find better combinations
   ```

In [ ]:
# Simulate multi-task scenario
def simulate_multi_task_performance(
    layer: MultiLoRALinear,
    x: torch.Tensor,
    task_targets: List[torch.Tensor],  # Target outputs for each task
    adapter_weights: List[float]
) -> Tuple[float, List[float]]:
    """
    Simulate performance of merged model on multiple tasks
    
    Returns:
        average_loss: Average loss across tasks
        task_losses: Individual task losses
    """
    # Get merged output
    merged_output = layer(x, adapter_weights)
    
    # Compute loss for each task
    task_losses = []
    for i, target in enumerate(task_targets):
        # Simulate: compute how far merged output is from task-specific target
        # Get output with just this task's adapter
        task_only_weights = [0.0] * len(adapter_weights)
        task_only_weights[i] = 1.0
        task_only_output = layer(x, task_only_weights)
        
        # Loss: distance from task-specific output
        loss = F.mse_loss(merged_output, task_only_output)
        task_losses.append(loss.item())
    
    avg_loss = np.mean(task_losses)
    return avg_loss, task_losses


# Create test scenario
layer = MultiLoRALinear(512, 512, rank=8, num_adapters=3)
x = torch.randn(8, 32, 512)

# Generate dummy targets (in practice, these would be from validation data)
task_names = ['Sentiment', 'NER', 'QA']
task_targets = [
    torch.randn(8, 32, 512) for _ in range(3)
]

# Test different weighting schemes
weighting_schemes = {
    'Uniform': [1/3, 1/3, 1/3],
    'Task 1 focused': [0.7, 0.2, 0.1],
    'Task 2 focused': [0.1, 0.7, 0.2],
    'Task 3 focused': [0.1, 0.2, 0.7],
    'Balanced': [0.4, 0.4, 0.2],
    'Equal top 2': [0.5, 0.5, 0.0],
}

print("Multi-Task Weighting Analysis:\n")
print("="*80)

results = []
for scheme_name, weights in weighting_schemes.items():
    avg_loss, task_losses = simulate_multi_task_performance(
        layer, x, task_targets, weights
    )
    results.append({
        'scheme': scheme_name,
        'weights': weights,
        'avg_loss': avg_loss,
        'task_losses': task_losses
    })
    
    print(f"\n{scheme_name}:")
    print(f"  Weights: {weights}")
    print(f"  Task losses: {[f'{l:.4f}' for l in task_losses]}")
    print(f"  Average: {avg_loss:.4f}")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap of weights
weights_matrix = np.array([r['weights'] for r in results])
scheme_names = [r['scheme'] for r in results]

im = ax1.imshow(weights_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
ax1.set_xticks(range(len(task_names)))
ax1.set_xticklabels(task_names)
ax1.set_yticks(range(len(scheme_names)))
ax1.set_yticklabels(scheme_names)
ax1.set_title('Adapter Weights by Scheme', fontsize=14, fontweight='bold')

# Add text annotations
for i in range(len(scheme_names)):
    for j in range(len(task_names)):
        text = ax1.text(j, i, f'{weights_matrix[i, j]:.2f}',
                       ha='center', va='center', color='black', fontsize=10)

plt.colorbar(im, ax=ax1, label='Weight')

# Task-specific performance
task_losses_matrix = np.array([r['task_losses'] for r in results])
x_pos = np.arange(len(scheme_names))
width = 0.25

for i, task_name in enumerate(task_names):
    offset = (i - 1) * width
    ax2.bar(x_pos + offset, task_losses_matrix[:, i], width, 
            label=task_name, alpha=0.8)

ax2.set_xlabel('Weighting Scheme', fontsize=11)
ax2.set_ylabel('Task Loss (lower better)', fontsize=11)
ax2.set_title('Performance by Task', fontsize=14, fontweight='bold')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(scheme_names, rotation=45, ha='right')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Find best scheme
best_idx = np.argmin([r['avg_loss'] for r in results])
best_scheme = results[best_idx]
print(f"\n" + "="*80)
print(f"Best scheme: {best_scheme['scheme']}")
print(f"Weights: {best_scheme['weights']}")
print(f"Average loss: {best_scheme['avg_loss']:.4f}")

## 5. Sequential vs Parallel Adapter Application

### Sequential Application
```
Apply adapters one after another:
  x1 = layer(x, adapter_1)
  x2 = layer(x1, adapter_2)  # Uses output of adapter_1
  x3 = layer(x2, adapter_3)  # Uses output of adapter_2

Effect: Adapters build on each other
Order matters! adapter_1 → adapter_2 ≠ adapter_2 → adapter_1
```

### Parallel Application
```
Apply all adapters to same input:
  x1 = layer(x, adapter_1)
  x2 = layer(x, adapter_2)  # Uses original x
  x3 = layer(x, adapter_3)  # Uses original x
  
  # Combine outputs
  x_final = α1*x1 + α2*x2 + α3*x3

Effect: Independent adapter contributions
Order doesn't matter
```

### Which to Use?

**Sequential:**
- Tasks are hierarchical (e.g., tokenize → parse → analyze)
- Later tasks depend on earlier outputs
- Pipeline-style processing
- Example: NER → Relation Extraction → Question Answering

**Parallel:**
- Tasks are independent
- Want to merge into single model
- More efficient (single forward pass)
- Example: Sentiment + Topic + Language Detection

**Hybrid:**
- Some adapters in parallel, some sequential
- Example: [Adapter_A || Adapter_B] → Adapter_C
  (A and B parallel, then C sequential)

In [ ]:
# Compare sequential vs parallel
class SequentialLoRA(nn.Module):
    """Apply LoRA adapters sequentially"""
    
    def __init__(self, in_features, out_features, rank, num_adapters):
        super().__init__()
        self.adapters = nn.ModuleList([
            LoRALinear(in_features, out_features, rank)
            for _ in range(num_adapters)
        ])
    
    def forward(self, x):
        """Apply adapters sequentially"""
        for adapter in self.adapters:
            x = adapter(x)
        return x


class ParallelLoRA(nn.Module):
    """Apply LoRA adapters in parallel and combine"""
    
    def __init__(self, in_features, out_features, rank, num_adapters):
        super().__init__()
        self.adapters = nn.ModuleList([
            LoRALinear(in_features, out_features, rank)
            for _ in range(num_adapters)
        ])
        # Learnable combination weights
        self.combination_weights = nn.Parameter(
            torch.ones(num_adapters) / num_adapters
        )
    
    def forward(self, x):
        """Apply adapters in parallel and combine"""
        outputs = [adapter(x) for adapter in self.adapters]
        outputs = torch.stack(outputs, dim=0)  # (num_adapters, batch, seq, dim)
        
        # Weighted combination
        weights = F.softmax(self.combination_weights, dim=0)
        combined = torch.sum(
            outputs * weights.view(-1, 1, 1, 1),
            dim=0
        )
        return combined


# Compare
in_features = 512
out_features = 512
rank = 8
num_adapters = 3

sequential_model = SequentialLoRA(in_features, out_features, rank, num_adapters)
parallel_model = ParallelLoRA(in_features, out_features, rank, num_adapters)

x = torch.randn(4, 32, in_features)

print("Sequential vs Parallel Adapter Application:\n")
print("="*70)

# Sequential
print("\n1. Sequential Application:")
out_seq = sequential_model(x)
print(f"   Output shape: {out_seq.shape}")
print(f"   Output mean: {out_seq.mean().item():.4f}")
print(f"   Output std: {out_seq.std().item():.4f}")
print("   Order: x → Adapter1 → Adapter2 → Adapter3")
print("   Each adapter modifies the output of previous")

# Parallel
print("\n2. Parallel Application:")
out_par = parallel_model(x)
print(f"   Output shape: {out_par.shape}")
print(f"   Output mean: {out_par.mean().item():.4f}")
print(f"   Output std: {out_par.std().item():.4f}")
weights = F.softmax(parallel_model.combination_weights, dim=0)
print(f"   Combination weights: {weights.detach().numpy()}")
print("   All adapters process same input, outputs combined")

# Timing comparison
import time

print("\n3. Performance Comparison:")
num_runs = 100

# Sequential timing
start = time.time()
for _ in range(num_runs):
    _ = sequential_model(x)
seq_time = (time.time() - start) / num_runs

# Parallel timing
start = time.time()
for _ in range(num_runs):
    _ = parallel_model(x)
par_time = (time.time() - start) / num_runs

print(f"   Sequential time: {seq_time*1000:.3f} ms/forward")
print(f"   Parallel time: {par_time*1000:.3f} ms/forward")
print(f"   Speedup: {seq_time/par_time:.2f}x")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Architecture comparison
ax1.axis('off')
arch_text = """
SEQUENTIAL:                    PARALLEL:

    Input x                        Input x
       |                              |
       v                    +---------+---------+
  Adapter 1                 |         |         |
       |                    v         v         v
       v                Adapter1  Adapter2  Adapter3
  Adapter 2                 |         |         |
       |                    +---------+---------+
       v                              |
  Adapter 3                           v
       |                      Weighted Combine
       v                              |
    Output                         Output

Order matters               Order doesn't matter
3 forward passes            Parallelizable
Compositional               Independent
"""
ax1.text(0.1, 0.5, arch_text, fontsize=11, family='monospace',
         verticalalignment='center',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
ax1.set_title('Architecture Comparison', fontsize=14, fontweight='bold', pad=20)

# Performance metrics
metrics = ['Forward Time', 'Compositionality', 'Parallelizability', 'Order Sensitivity']
sequential_scores = [3, 10, 3, 10]  # Slower, compositional, not parallel, order matters
parallel_scores = [10, 5, 10, 2]    # Faster, less compositional, parallel, order invariant

x_pos = np.arange(len(metrics))
width = 0.35

bars1 = ax2.bar(x_pos - width/2, sequential_scores, width, 
                label='Sequential', color='#4ECDC4', alpha=0.8)
bars2 = ax2.bar(x_pos + width/2, parallel_scores, width,
                label='Parallel', color='#FF6B6B', alpha=0.8)

ax2.set_ylabel('Score (0-10)', fontsize=11)
ax2.set_title('Characteristic Comparison', fontsize=14, fontweight='bold')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(metrics, rotation=15, ha='right')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)
ax2.set_ylim([0, 11])

plt.tight_layout()
plt.show()

print("\nRecommendation:")
print("  Sequential: Use for hierarchical/dependent tasks")
print("  Parallel: Use for independent tasks, more efficient")

## 6. Model Soups with LoRA

### Concept
```
Model Soup (Wortsman et al., 2022):
  Train multiple models on same task with different:
    - Random seeds
    - Hyperparameters  
    - Training data order
  
  Average the models:
    W_soup = (1/n) Σ W_i
  
  Result: Often better than best individual model!
```

### LoRA Model Soups
```
Even better with LoRA:
  - Train k LoRA adapters with different random seeds
  - Each adapter: BA_i
  - Merge: W_soup = W_base + (1/k) Σ(BA_i)
  
Benefits:
  - Reduces variance from random initialization
  - Smoother loss landscape
  - More robust predictions
  - Often 1-2% accuracy improvement
```

### Greedy Soup
```
Instead of uniform averaging, greedily add models:
  1. Start with best individual model
  2. For each remaining model:
     - Try adding to soup
     - Keep if validation improves
  3. Result: Soup with only helpful models
```

In [ ]:
# Simulate model soup
def create_lora_soup(
    base_layer: nn.Module,
    num_models: int = 5,
    in_features: int = 512,
    out_features: int = 512,
    rank: int = 8
) -> Dict[str, torch.Tensor]:
    """
    Create LoRA model soup by averaging multiple adapters
    """
    # Train multiple LoRA adapters (simulated with different random seeds)
    adapters = []
    for seed in range(num_models):
        torch.manual_seed(seed)
        adapter = LoRALinear(in_features, out_features, rank)
        adapters.append(adapter)
    
    # Compute average LoRA parameters
    avg_A = torch.stack([a.lora_A.data for a in adapters]).mean(dim=0)
    avg_B = torch.stack([a.lora_B.data for a in adapters]).mean(dim=0)
    
    # Also compute variance (for analysis)
    std_A = torch.stack([a.lora_A.data for a in adapters]).std(dim=0)
    std_B = torch.stack([a.lora_B.data for a in adapters]).std(dim=0)
    
    return {
        'adapters': adapters,
        'avg_A': avg_A,
        'avg_B': avg_B,
        'std_A': std_A,
        'std_B': std_B
    }


# Create soup
print("Model Soup Creation:\n")
print("="*70)

base = nn.Linear(512, 512)
soup_data = create_lora_soup(base, num_models=5)

print(f"\nCreated soup from {len(soup_data['adapters'])} LoRA adapters")
print(f"\nAverage LoRA A:")
print(f"  Shape: {soup_data['avg_A'].shape}")
print(f"  Mean: {soup_data['avg_A'].mean().item():.6f}")
print(f"  Std across models: {soup_data['std_A'].mean().item():.6f}")

print(f"\nAverage LoRA B:")
print(f"  Shape: {soup_data['avg_B'].shape}")
print(f"  Mean: {soup_data['avg_B'].mean().item():.6f}")
print(f"  Std across models: {soup_data['std_B'].mean().item():.6f}")

# Test on sample input
x_test = torch.randn(4, 32, 512)

# Individual adapter outputs
individual_outputs = [adapter(x_test) for adapter in soup_data['adapters']]
individual_mean = torch.stack(individual_outputs).mean(dim=0)
individual_std = torch.stack(individual_outputs).std(dim=0)

# Soup adapter output
soup_adapter = LoRALinear(512, 512, rank=8)
soup_adapter.lora_A.data = soup_data['avg_A']
soup_adapter.lora_B.data = soup_data['avg_B']
soup_output = soup_adapter(x_test)

print("\n" + "="*70)
print("Output Analysis:")
print(f"\nIndividual models:")
print(f"  Mean of outputs: {individual_mean.mean().item():.6f}")
print(f"  Std across models: {individual_std.mean().item():.6f}")

print(f"\nSoup model:")
print(f"  Mean of output: {soup_output.mean().item():.6f}")
print(f"  Diff from average: {(soup_output - individual_mean).abs().mean().item():.8f}")

# Visualize variance reduction
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Output distributions
for i, adapter in enumerate(soup_data['adapters']):
    out = adapter(x_test)
    ax1.hist(out.flatten().detach().numpy(), bins=50, alpha=0.3, 
            label=f'Model {i+1}', density=True)

ax1.hist(soup_output.flatten().detach().numpy(), bins=50, alpha=0.8,
        label='Soup', color='red', density=True, linewidth=2, histtype='step')
ax1.set_xlabel('Output Value', fontsize=11)
ax1.set_ylabel('Density', fontsize=11)
ax1.set_title('Output Distributions: Individual vs Soup', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Variance comparison
positions = range(5)
individual_vars = [out.var().item() for out in individual_outputs]
soup_var = soup_output.var().item()

bars = ax2.bar(positions, individual_vars + [soup_var],
              color=['#4ECDC4']*5 + ['#FF6B6B'],
              alpha=0.8, edgecolor='black')
ax2.set_xticks(range(6))
ax2.set_xticklabels([f'M{i+1}' for i in range(5)] + ['Soup'])
ax2.set_ylabel('Output Variance', fontsize=11)
ax2.set_title('Variance: Individual Models vs Soup', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# Highlight soup
ax2.axhline(y=soup_var, color='red', linestyle='--', alpha=0.5, label='Soup level')
ax2.legend()

plt.tight_layout()
plt.show()

print("\nKey Insight: Soup model has smoother, more stable outputs")
print("             Reduces variance from random initialization")

## Summary: LoRA Merging Strategies

### Key Techniques

1. **Deployment Merging**
   - Merge LoRA into base: W' = W + BA
   - Zero inference overhead
   - Essential for production

2. **Task Arithmetic**
   - Add: W + BA1 + BA2 (multi-task)
   - Subtract: W + BA_good - BA_bad (remove behavior)
   - Average: W + (1/n)Σ(BA_i) (ensemble)
   - Interpolate: W + α*BA1 + (1-α)*BA2

3. **Weighted Merging**
   - Uniform: αi = 1/k
   - Performance-based: αi ∝ accuracy_i
   - Learned: α = MLP(input)
   - Task-specific: Different α per sample

4. **Sequential vs Parallel**
   - Sequential: For hierarchical tasks
   - Parallel: For independent tasks (more efficient)
   - Hybrid: Mix both approaches

5. **Model Soups**
   - Train multiple with different seeds
   - Average adapters: W + (1/k)Σ(BA_i)
   - Reduces variance, improves robustness
   - 1-2% accuracy gain typical

### Practical Recommendations

**For Production:**
- Always merge for deployment (zero overhead)
- Test merged model equals unmerged
- Can unmerge if needed

**For Multi-Task:**
- Start with uniform weighting (baseline)
- Optimize weights on validation set
- Consider task-specific routing

**For Robustness:**
- Train 3-5 models with different seeds
- Create model soup by averaging
- Usually outperforms best individual

**For Continual Learning:**
- Keep separate adapters per task
- Merge with careful weighting
- Monitor for catastrophic forgetting

### Next Steps
- Notebook 29: Advanced LoRA variants (DoRA, AdaLoRA, LoRA+, QA-LoRA)

### References
- Wortsman et al. (2022): "Model soups: averaging weights of multiple fine-tuned models"
- Ilharco et al. (2022): "Editing Models with Task Arithmetic"
- Pfeiffer et al. (2020): "AdapterFusion: Non-Destructive Task Composition"
- HuggingFace PEFT: https://github.com/huggingface/peft